#Mount to drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Imports

In [ ]:
import csv, shutil, tarfile, hashlib
from pathlib import Path
from google.colab import files
import os
import tempfile

# Functions

In [ ]:
TAR_EXTS = (".tar", ".tar.gz", ".tgz")  # can add ".tar.bz2", ".tbz2", etc.
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
PROTO_NAME = "protocol"  # exact filename inside archive (case-insensitive)


def is_tar(p: Path) -> bool:
    if not p.is_file():
        return False
    lower_name = p.name.lower()
    return any(lower_name.endswith(ext) for ext in TAR_EXTS)


def read_protocol_text(tar_path: Path) -> str:
    try:
        with tarfile.open(tar_path, "r:*") as t:
            for m in t.getmembers():
                if m.isfile() and Path(m.name).name.lower() == PROTO_NAME:
                    fh = t.extractfile(m)
                    if fh:
                        try:
                            return fh.read().decode("utf-8", errors="ignore")
                        finally:
                            fh.close()
    except Exception:
        return ""
    return ""

# Upload the TAR files
You can just controll+A inside the Raw data at the date you want to do, but it needs to be temporarily uploaded in order that Colab can access it, it will be deleted soon after closing this Colab file.

In [ ]:
temp_upload_dir = tempfile.mkdtemp(dir='/content')
print(f"Temporary upload directory created at: {temp_upload_dir}")

uploaded = files.upload(target_dir=temp_upload_dir)
for fn in uploaded.keys():
  print('"{name}" with length {length} bytes'.format(name=fn, length=len(uploaded[fn])))

# Sorting Script
Modify the code to set the output directory to `/content/drive/MyDrive/Sorted`, run the modified code, zip the output directory, and provide a download link for the zipped file.

In [ ]:
# Update these paths to the actual location of your files in Colab
INPUT_DIR = temp_upload_dir
OUTPUT_DIR = "/content/drive/MyDrive/Sorted"

in_root = Path(INPUT_DIR).resolve()

# Check if the input directory exists
if not in_root.exists():
    print(f"Error: Input directory not found at {in_root}")
else:
    all_files = [p for p in in_root.rglob("*") if p.is_file()]
    image_files = [p for p in all_files if p.suffix.lower() in IMG_EXTS]
    tar_files = [p for p in all_files if is_tar(p)]

    print("\n  File Overview ")
    print(f"Total files found: {len(all_files)}")
    print(f"Image files:       {len(image_files)}")
    print(f"TAR archives:      {len(tar_files)}")

    out_root = Path(OUTPUT_DIR).resolve()
    out_root.mkdir(parents=True, exist_ok=True)

    files = sorted(tar_files)

    group_map = {}
    next_id = 1
    rows = []

    for f in files:
        text = read_protocol_text(f)
        if not text:
            group = "Protocol_MISSING"
        else:
            key = hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()
            if key not in group_map:
                group_map[key] = f"Protocol_{next_id}"
                next_id += 1
            group = group_map[key]

        dest = out_root / group / f.name
        dest.parent.mkdir(parents=True, exist_ok=True)

        try:
            shutil.copy2(f, dest)
        except Exception:
            pass

        rows.append({
            "TarFileName": f.name,
            "ProtocolGroup": group
        })

    if rows:
        csv_path = out_root / "protocol_grouping.csv"
        with csv_path.open("w", newline="", encoding="utf-8") as fp:
            writer = csv.DictWriter(fp, fieldnames=["TarFileName", "ProtocolGroup"])
            writer.writeheader()
            writer.writerows(rows)
        print(f"Done. CSV saved at: {csv_path}")
    else:
        print("No TAR files found.")

## Zip the output directory

### Subtask:
Zip the output directory containing the sorted files and the CSV and download that zip via a link.


In [ ]:
zip_output_path = os.path.join(temp_upload_dir, 'sorted_files')
shutil.make_archive(zip_output_path, 'zip', OUTPUT_DIR)
print(f"Zip file created at: {zip_output_path}.zip")

files.download(f"{zip_output_path}.zip")

# Delete uploaded files from the Colab environment
**These files still need to be deleted from the Trashbin in Google Drive, by hand, check that not the wrong files are added to it like notebooks.**

In [ ]:
# Delete the entire temporary upload directory
if 'temp_upload_dir' in globals() and os.path.exists(temp_upload_dir):
    try:
        shutil.rmtree(temp_upload_dir)
        print(f"Deleted temporary upload directory: {temp_upload_dir}")
    except OSError as e:
        print(f"Error deleting temporary upload directory {temp_upload_dir}: {e}")
else:
    print(f"Temporary upload directory not found or not defined: {temp_upload_dir if 'temp_upload_dir' in globals() else 'variable not defined'}")

# Note: Deletion from Google Drive itself needs to be done manually.
INPUT_DIR = "/content"
in_root = Path(INPUT_DIR).resolve()
drive_path = Path('/content/drive').resolve() # Define the Google Drive path

if in_root.exists():
    # Re-find all files in the input directory
    all_files = [p for p in in_root.rglob("*") if p.is_file()]
    # Exclude files in sample_data directory, the .config directory, and the Google Drive directory
    files_to_delete = [f for f in all_files if not any(part.startswith('.') or part == 'sample_data' for part in f.parts) and not drive_path in f.parents]

    print(f"\nAttempting to delete {len(files_to_delete)} files from {INPUT_DIR} (excluding files in .config, sample_data, and Google Drive):")
    for f in files_to_delete:
        try:
            os.remove(f)
            print(f"Deleted {f} from Colab environment.")
        except OSError as e:
            print(f"Error deleting {f}: {e}")
else:
    print(f"Input directory not found at {in_root}, cannot delete other files.")